Intermediate: Annual trade by NACIS classification
==================================================

In [1]:
from time import sleep
from pathlib import Path 
import pandas as pd 
import sys

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
import duckdb as dd

from utils.paths import (STG_STATCAN_DIR,
                         INT_MACROECON_FLOWS_DIR)

In [3]:
trade_by_naics_filename = 'stg_statcan__trade_by_naics.parquet'


In [2]:
# File & Path params
IN_TBL_NAME = "stg_statcan__trade_by_naics"
IN_FILEPATH = STG_STATCAN_DIR / f"{IN_TBL_NAME}.parquet"

OUT_TBL_NAME = "int_trade_by_naics"
OUT_FILEPATH = INT_MACROECON_FLOWS_DIR / f"{OUT_TBL_NAME}.parquet"

# Regex patterns for NAICS parsing
industry_grp_regexp = r"\[(\d{4})\]$"
industry_grp_name_regexp = r"\s*\[[^\]]+\]$"

In [4]:
con = dd.connect()

In [5]:
int_query = f"""
-- Minimal intermediate: yearly trade by NAICS
-- Keeps only: year, trade_type, trading_partner, naics_code, naics_name, cad_value

WITH naics_parsed AS (
    SELECT
        year,
        trade_type,
        trading_partner,

        -- Clean NAICS name
        CASE
            WHEN naics = 'Total of all industries' THEN 'Total'
            ELSE TRIM(REGEXP_REPLACE(naics, '{industry_grp_name_regexp}', ''))
        END AS naics_name,

        -- Numeric NAICS code (NULL for Total)
        CASE
            WHEN REGEXP_MATCHES(naics, '{industry_grp_regexp}')
            THEN TRY_CAST(REGEXP_EXTRACT(naics, '{industry_grp_regexp}', 1) AS INT)
            ELSE NULL
        END AS naics_code,

        cad_value
    FROM
        read_parquet('{IN_FILEPATH}')
)
SELECT
    year,
    trade_type,
    trading_partner,
    naics_code,
    naics_name,
    SUM(cad_value) AS cad_value
FROM
    naics_parsed
GROUP BY ALL
"""

In [6]:
# Create view and materialize as parquet
con.sql(f"CREATE OR REPLACE VIEW '{OUT_TBL_NAME}' AS ({int_query})")

save_query = f"""
COPY
    (SELECT * FROM '{OUT_TBL_NAME}')
TO
    '{OUT_FILEPATH.as_posix()}'
(FORMAT PARQUET);
"""

con.execute(save_query)
con.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))